# Average sim realizations into combined P / B template files

`PS_Sim.sh` / `BS_Sim.sh` measure each synthetic field per realization, writing `*_ps_v{v}.dat` and `*_comb_v{v}.dat` (`v = 1..25`). The fitting driver (`Sim_BiasFitting/Full_Parallel_Fits.py`) instead reads a single *combined* template per term (`b1b1_ps.dat`, `b1b1b2_comb.dat`, ...). This notebook produces those combined files by averaging the realizations.

Method (reverse-engineered from the byte-exact `OLD_PowerSpectra` / `OLD_Bispectra` files):
- average **only** the spectrum column (`P_bin` for power, `B_bin` for bispectrum) across the realizations,
- copy every other column (k-bins, `N_modes`, `triangle_index`, ...) verbatim from realization 1,
- write in the original BSKIT `.dat` layout: header copied verbatim, `str()`-formatted floats, integer `triangle_index`, one trailing space per row.

In [1]:
import re
import glob
import numpy as np
import pandas as pd

SIMS = '/mnt/nvme1/kboone/Amplitude_Sims/'
POWER_DIR = SIMS + 'PowerSpectra/'
BISPEC_DIR = SIMS + 'Bispectra/'

In [2]:
def load(f):
    # Same header-offset fix as Fits_Helpers: roll columns left, drop the dangling last one.
    df = pd.read_csv(f, sep=r'\s+')
    df.columns = np.roll(df.columns, -1)
    return df.drop(df.columns[-1], axis=1)


def realizations(prefix, folder):
    """All `{prefix}_v{N}.dat` files in folder, sorted by realization number."""
    pat = re.compile(re.escape(prefix) + r'_v(\d+)\.dat$')
    fs = [f for f in glob.glob(folder + prefix + '_v*.dat') if pat.search(f)]
    return sorted(fs, key=lambda f: int(pat.search(f).group(1)))


def discover(folder, suffix):
    """Unique prefixes (e.g. 'b1b1_ps') of files named `{prefix}_v{N}.dat`, where prefix ends with suffix."""
    pat = re.compile(r'([^/]+' + re.escape(suffix) + r')_v\d+\.dat$')
    prefixes = {m.group(1) for f in glob.glob(folder + '*_v*.dat') if (m := pat.search(f))}
    return sorted(prefixes)


def write_combined(out_path, header, table, int_first_col):
    with open(out_path, 'w') as f:
        f.write(header)
        for row in table:
            first = str(int(round(row[0]))) if int_first_col else str(float(row[0]))
            vals = [first] + [str(float(x)) for x in row[1:]]
            f.write(' '.join(vals) + ' \n')


def combine(prefix, folder, spec_col, int_first_col):
    fs = realizations(prefix, folder)
    dfs = [load(f) for f in fs]
    shapes = {d.shape for d in dfs}
    assert len(shapes) == 1, f'{prefix}: realizations disagree on shape: {shapes}'
    spec = np.array([d[spec_col].to_numpy() for d in dfs])

    # Sanity flag: a realization whose overall scale is wildly off the rest (cf. the corrupted
    # b1b1F2 v1 in OLD_Bispectra). Only warns -- the mean still uses every realization.
    scale = np.median(np.abs(spec), axis=1)
    med = np.median(scale)
    odd = np.where((scale > 5 * med) | (scale < med / 5))[0]
    if len(odd):
        print(f'  WARNING {prefix}: outlier realizations (1-indexed) {odd + 1}; '
              f'median|spec| {np.round(scale, 1)}')

    table = dfs[0].to_numpy().copy()              # k-bins, N_modes, etc. from realization 1
    j = list(dfs[0].columns).index(spec_col)
    table[:, j] = spec.mean(axis=0)               # average the spectrum column only

    with open(fs[0]) as f:
        header = f.readline()
    out_path = folder + prefix + '.dat'
    write_combined(out_path, header, table, int_first_col)
    print(f'  {prefix}: averaged {len(fs)} realizations -> {out_path}')

## Power spectra (`*_ps_v{N}.dat` -> `*_ps.dat`)

In [3]:
for prefix in discover(POWER_DIR, '_ps'):
    combine(prefix, POWER_DIR, 'P_bin_[h^-3_Mpc^3]', int_first_col=False)

  F2F2_ps: averaged 25 realizations -> /mnt/nvme1/kboone/Amplitude_Sims/PowerSpectra/F2F2_ps.dat
  K2F2_ps: averaged 25 realizations -> /mnt/nvme1/kboone/Amplitude_Sims/PowerSpectra/K2F2_ps.dat
  K2K2_ps: averaged 25 realizations -> /mnt/nvme1/kboone/Amplitude_Sims/PowerSpectra/K2K2_ps.dat
  b1F3_ps: averaged 25 realizations -> /mnt/nvme1/kboone/Amplitude_Sims/PowerSpectra/b1F3_ps.dat
  b1K2_2_ps: averaged 25 realizations -> /mnt/nvme1/kboone/Amplitude_Sims/PowerSpectra/b1K2_2_ps.dat
  b1Otd_ps: averaged 25 realizations -> /mnt/nvme1/kboone/Amplitude_Sims/PowerSpectra/b1Otd_ps.dat
  b1b1_ps: averaged 25 realizations -> /mnt/nvme1/kboone/Amplitude_Sims/PowerSpectra/b1b1_ps.dat
  b1bk2_ps: averaged 25 realizations -> /mnt/nvme1/kboone/Amplitude_Sims/PowerSpectra/b1bk2_ps.dat
  b2F2_ps: averaged 25 realizations -> /mnt/nvme1/kboone/Amplitude_Sims/PowerSpectra/b2F2_ps.dat
  b2K2_ps: averaged 25 realizations -> /mnt/nvme1/kboone/Amplitude_Sims/PowerSpectra/b2K2_ps.dat
  b2b2_ps: averaged 25

## Bispectra (`*_comb_v{N}.dat` -> `*_comb.dat`)

Only the normalized `_comb` files are combined; the intermediate `_unnormbs` realizations are ignored.

In [4]:
for prefix in discover(BISPEC_DIR, '_comb'):
    combine(prefix, BISPEC_DIR, 'B_bin_[h^-6_Mpc^6]', int_first_col=True)

  b1b1F2_comb: averaged 25 realizations -> /mnt/nvme1/kboone/Amplitude_Sims/Bispectra/b1b1F2_comb.dat
  b1b1K2_comb: averaged 25 realizations -> /mnt/nvme1/kboone/Amplitude_Sims/Bispectra/b1b1K2_comb.dat
  b1b1b2_comb: averaged 25 realizations -> /mnt/nvme1/kboone/Amplitude_Sims/Bispectra/b1b1b2_comb.dat
  b1b2b3_comb: averaged 25 realizations -> /mnt/nvme1/kboone/Amplitude_Sims/Bispectra/b1b2b3_comb.dat
  b2b2b2_comb: averaged 25 realizations -> /mnt/nvme1/kboone/Amplitude_Sims/Bispectra/b2b2b2_comb.dat


## Check

Reload a combined file through the same loader the fitting code uses, to confirm it parses cleanly.

In [5]:
for f in [POWER_DIR + 'b1b1_ps.dat', BISPEC_DIR + 'b1b1b2_comb.dat']:
    df = load(f)
    print(f, '->', df.shape, list(df.columns))

/mnt/nvme1/kboone/Amplitude_Sims/PowerSpectra/b1b1_ps.dat -> (127, 4) ['k_mean_over_bin_[h_Mpc^-1]', 'P_bin_[h^-3_Mpc^3]', 'N_modes', 'Shot_noise']
/mnt/nvme1/kboone/Amplitude_Sims/Bispectra/b1b1b2_comb.dat -> (17611, 12) ['triangle_index_[dimensionless]', 'k1_mean_[h_Mpc^-1]', 'k2_mean_[h_Mpc^-1]', 'k3_mean_[h_Mpc^-1]', 'k1_min_[h_Mpc^-1]', 'k1_max_[h_Mpc^-1]', 'k2_min_[h_Mpc^-1]', 'k2_max_[h_Mpc^-1]', 'k3_min_[h_Mpc^-1]', 'k3_max_[h_Mpc^-1]', 'B_bin_[h^-6_Mpc^6]', 'N_modes']
